Chapter 5 - Pretraining on Unlabeled Data

5.1.1 - Using GPT to generate text

In [1]:
import torch

In [2]:
from previous_chapters import GPTModel

In [3]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [4]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [5]:
import tiktoken
from previous_chapters import generate_text_simple

In [10]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

In [7]:
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [8]:
start_context = "Every effort moves you"

In [9]:
tokenizer = tiktoken.get_encoding("gpt2")


In [11]:
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

In [12]:
token_ids_to_text(token_ids, tokenizer)

'Every effort moves you rentingetic wasnم refres RexMeCHicular stren'

5.1.2 - Calculating the text generation loss: cross-entropy and perplexity

In [13]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [14]:
with torch.no_grad():
    logits = model(inputs)

In [15]:
probas = torch.softmax(logits, dim=-1)

In [17]:
probas.shape

torch.Size([2, 3, 50257])

In [19]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)

In [20]:
token_ids

tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])

In [21]:
token_ids_to_text(targets[0], tokenizer)

' effort moves you'

In [22]:
token_ids_to_text(token_ids[0].flatten(), tokenizer)

' Armed heNetflix'

In [32]:
token_ids[0].flatten().shape

torch.Size([3])

In [36]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
target_probas_1

tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])

In [37]:
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
target_probas_2

tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])

In [38]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
log_probas

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])

In [41]:
avg_log_probas = torch.mean(log_probas)
avg_log_probas

tensor(-10.7940)

In [42]:
neg_avg_log_probas = avg_log_probas * -1
neg_avg_log_probas

tensor(10.7940)

In [43]:
logits.shape

torch.Size([2, 3, 50257])

In [44]:
targets.shape

torch.Size([2, 3])

In [48]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

In [49]:
logits_flat.shape, targets_flat.shape

(torch.Size([6, 50257]), torch.Size([6]))

In [50]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
loss

tensor(10.7940)

In [51]:
perplexity = torch.exp(loss)
perplexity

tensor(48725.8203)